# DocuMind: Enterprise Document Q&A system

This notebook builds the foundation of a document intelligence system.

In this stage, we will:
1. Load PDF documents
2. Split them into smaller text chunks
3. Convert chunks into embeddings
4. Store them in a FAISS vector database
5. Test retrieval using a sample question

This forms the backbone of a Retrieval-Augmented Generation (RAG) pipeline.

## 1. Install required Libraries & Packages

In [1]:
#installing packages

!pip install langchain langchain-community langchain-text-splitters
!pip install faiss-cpu pypdf sentence-transformers
!pip install mlflow

In [2]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters
!pip install -U langchain langchain-core langchain-community langchain-text-splitters pypdf faiss-cpu

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.2.29
Uninstalling langchain-core-1.2.29:
  Successfully uninstalled langchain-core-1.2.29
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-text-splitters 1.1.1
Uninstalling langchain-text-splitters-1.1.1:
  Successfully uninstalled langchain-text-splitters-1.1.1
  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.29-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langchain_core-1.2.29-py3-none-any.whl (508 kB)
Using cached langch

import langchain
import langchain_core
import langchain_community
import langchain_text_splitters

In [3]:
import nltk
print(nltk.__file__)
print("NLTK import ok")

/opt/anaconda3/envs/gouthami/lib/python3.11/site-packages/nltk/__init__.py
NLTK import ok


In [4]:
# import required libraries
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

import mlflow

## 2. Load PDF Documents

In this step, we will read each pdf file and convert it into LangChain 'Document' objects

Each page, becomes a document object containing:
- page content
- metadata such as page number and source file

In [5]:
DATA_DIR = Path("data/sample_docs")

all_docs =[]
pdf_files = list(DATA_DIR.glob("*.pdf"))

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    docs = loader.load()
    all_docs.extend(docs)

print(f"Total number of loaded pages: {len(all_docs)}")

Total number of loaded pages: 34


## 3. Inspecting the loaded documents

This step is crucial, as it helps confirm that the PDF was read correctly

In [6]:
if all_docs:
    print("Sample page content: \n")
    print(all_docs[0].page_content[1000:2000])
    print("\nMeta Data: \n")
    print(all_docs[0].metadata)
else: print("No document were loaded")

Sample page content: 

periments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and
has been cruci

## 4. Chunking Strategy

Large documents are not stored as 1 long block of text.


Instead, we split them into smaller chunks so that: 
- retrival becomes more accurate
- the LLM receives only the accurate information
- large documents fit within model limits

We start with:
- chunk size = 500
- chunk overlap = 50

In [47]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 20
)
print("Text splitter created successfully")

Text splitter created successfully


In [48]:
#now lets split documents into chunks

split_docs = text_splitter.split_documents(all_docs)
print(f"Total number of chunks created: {len(split_docs)}")

#embeddings work better on shorter, focused pieces of text

Total number of chunks created: 137


In [49]:
# inspecting the chunks

if split_docs:
    print("Sample chunk:\n")
    print(split_docs[0].page_content)

    print("\nChunk metadata:\n")
    print(split_docs[0].metadata)
else:
    print("No chunks created.")

Sample chunk:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutio

## 5. Setup MLflow Environment

We use MLflow to track our chunking experiments.

For this run, we will log:
- chunk size
- chunk overlap
- number of loaded pages
- number of chunks created

MLflow helps us keep a record of our experiments.

In [50]:
mlflow.set_experiment("documind_chunking_experiment")

<Experiment: artifact_location='/Users/gayathrichelluri/Documents/Gouthu/Projects/DocuMind/mlruns/1', creation_time=1776229843640, experiment_id='1', last_update_time=1776229843640, lifecycle_stage='active', name='documind_chunking_experiment', tags={}, trace_location=None, workspace='default'>

In [51]:
with mlflow.start_run(run_name=f"chunk_{500}_overlap_{50}"):
    mlflow.log_param("chunk_size", 500)
    mlflow.log_param("chunk_overlap", 50)
    mlflow.log_metric("num_loaded_pages", len(all_docs))
    mlflow.log_metric("num_chunks_created", len(split_docs))

print("Chunking experiment logged successfully.")

Chunking experiment logged successfully.


## 6. Creating Embeddings

We will now convert each text chunk into a vector representation called an embedding

Embeddings help in measuring the semantic similarity between user questions and document chunks

In [52]:
!pip install sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [53]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


In [54]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

## 7. Create FAISS Vector Store

Now we will be storing the chunk embeddings in FAISS

FAISS allows fast similarity search, so later when a user asks a question, we can retrive the most relevant chunks>

In [55]:
vector_store = FAISS.from_documents(
    documents=split_docs,
    embedding=embedding_model
)

print("FAISS vector store created successfully")

FAISS vector store created successfully


In [56]:
#now lets save the vector store for future use:
vector_store.save_local("faiss_index")
print("FAISS index saved locally")

FAISS index saved locally


## 8. Test Reterival with a sample question

Before connecting an LLM, we first test whether the retrival working correctly

In [57]:
query = "What is this document mainly about?

retrived_docs = vector_store.similarity_search(query, k =3)
print(f"Retrived {len(retrived_docs)} chunk. \n")

for i, doc in enumerate(retrived_docs, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.page_content[:500])
    print("\nMetadata:", doc.metadata)
    print("\n" + "="*80 + "\n")

Object `about` not found.
Retrived 3 chunk. 

--- Retrieved Chunk 1 ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-dista

Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/sample_docs/attention_all_you_need.pdf', 'total_pages': 15, 'p

## 9. Conclusion

In this notebook stage, we successfully:
- loaded PDF documents
- split them into smaller chunks
- generated embeddings
- stored them in a FAISS vector database
- logged chunking settings in MLflow
- tested similarity-based retrieval

This completes the ingestion and retrieval backbone of the DocuMind system.

The next stage will connect retrieval with an LLM to generate grounded answers using RAG.

# Stage 2 — RAG (Answer Generation)

## 1. Load vector store

In [58]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)
print("FAISS vector store loaded")

FAISS vector store loaded


## 2. Create Retriever

In [59]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

print('retriever created')

retriever created


## 3. Setup LLM

We use a local LLM via Ollama (LLaMA 3).

Make sure you have:
- Ollama installed
- model pulled: `ollama run llama3`

In [60]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")

print("LLM connected")

LLM connected


## 4. Prompt Templates

In [61]:
# Basic prompt
basic_prompt = """
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

# Chain-of-thought prompt
cot_prompt = """
You are a helpful assistant. Think step by step using the context.

Context:
{context}

Question:
{question}



"""

# Few-shot prompt
few_shot_prompt = """
Example:
Context: The company follows a hybrid work model.
Question: What work model is followed?
Answer: The company follows a hybrid work model.

Now answer the following:

Context:
{context}

Question:
{question}

Answer:
"""

## 5. RAG Function
This function retrieves the chunks and build the prompt and then sends it to the LLM

In [62]:
def rag(query, prompt_template):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = prompt_template.format(context=context,question = query)
    response = llm.invoke(prompt)

    return response,docs

## 6. Test with query

In [65]:
query = "What topics are discussed in this document?"

answer, docs = rag(query, cot_prompt)

print("Answer: \n")
print(answer)

Answer: 

I'd be happy to help!

After reading the context, I noticed that it appears to be a collection of research papers and conference proceedings related to Natural Language Processing (NLP) and Computational Linguistics. The specific topics being discussed seem to revolve around language models, text classification, machine comprehension, and sentence representation learning.

Some of the key concepts mentioned in the papers include:

1. Fine-tuning language models for various NLP tasks
2. Text classification and sentiment analysis
3. Machine reading comprehension and question answering
4. Sentence representation learning and contextualized word embeddings

Based on this information, I would categorize the topics discussed in this document as follows:

* Natural Language Processing (NLP)
* Computational Linguistics
* Language Models
* Text Classification
* Sentiment Analysis
* Machine Reading Comprehension
* Question Answering
* Sentence Representation Learning

Please let me kno